[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nattaponm/Teach_stormTracks_KKN_May2024_HID_doppler/blob/main/notebooks/05_terrain_range_reports.ipynb)


# บทที่ 05 — ภูมิประเทศ ระยะเรดาร์ และการตรวจสอบกับรายงานความเสียหาย

ภาคอีสานด้านตะวันตกมีแนวเขาเพชรบูรณ์-เลย ทางเหนือมีภูพาน บทนี้ถามว่า **ภูมิประเทศสัมพันธ์กับจุดกำเนิดและความรุนแรงของพายุหรือไม่** โดยระวังตัวแปรแทรกซ้อนสำคัญคือ **ระยะจากเรดาร์** (ภูเขาอยู่ห่างเรดาร์ 90–200 km) แล้วปิดท้ายด้วยการจับคู่เส้นทางกับรายงานความเสียหายจากบท 01

### วัตถุประสงค์การเรียนรู้
1. คำนวณและอ่านตัวแปรภูมิประเทศ (ความสูง ความลาดชัน TPI) และแรงยกจากภูมิประเทศตามทิศการเคลื่อนที่
2. ทดสอบว่าจุดกำเนิดพายุอยู่บนพื้นที่สูงกว่าที่คาดจากการสุ่มหรือไม่ (permutation test)
3. ใช้ partial correlation ควบคุมผลของระยะเรดาร์
4. จับคู่เส้นทางกับรายงานความเสียหายด้วยเงื่อนไขทั้งพื้นที่และเวลา แล้วเปรียบเทียบพายุที่ "สร้างความเสียหาย" กับพายุอื่น
5. เชื่อมสภาพแวดล้อมรายวันจาก sounding กับสถิติพายุรายวัน

In [ ]:
# @title ⚙️ เตรียมสภาพแวดล้อม (รันก่อนเสมอ)
import os, sys, subprocess
REPO = "Teach_stormTracks_KKN_May2024_HID_doppler"
if "google.colab" in sys.modules:
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", f"https://github.com/nattaponm/{REPO}.git", f"/content/{REPO}"], check=True)
    else:
        subprocess.run(["git", "-C", f"/content/{REPO}", "pull", "-q"])
    os.chdir(f"/content/{REPO}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) in ("notebooks", "prep"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("โฟลเดอร์ทำงาน:", os.getcwd(), "| CPU", os.cpu_count(), "core")

In [ ]:
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats, ndimage
warnings.filterwarnings("ignore")

from utils.config import load_settings, repo_path
from utils import maps as M, geo as G, data as D, analysis as A, track_tools as tt
s = load_settings()
M.setup_style()
pd.set_option("display.width", 200, "display.max_columns", 30)
DEM = G.load_dem(s)                                   # (array, extent) — สร้างจาก Copernicus ครั้งแรกถ้ายังไม่มี
idx = D.day_index()
KEY_DAYS = list(idx.loc[idx["in_repo"], "day"]) if len(idx) else [pd.Timestamp("2024-05-03")]
KEY = KEY_DAYS[0]                                     # วันหลักที่มีกริดใน repo (เปลี่ยนเป็นวันอื่นได้ จะดาวน์โหลดจาก Zenodo)
print("วันหลักใน repo:", [f"{d:%d %b}" for d in KEY_DAYS], "| ใช้วัน:", f"{KEY:%d %b %Y}")

import geopandas as gpd
TAG = "15min"
T = A.life_classes(A.month_tracks(TAG, min_life=s["tracking"]["min_life_min"]))
C = A.month_cells(TAG)
TER = D.terrain()
_, meta = D.load_cappi(KEY)
print(f"เส้นทาง {len(T)} เส้น | ตัวแปรภูมิประเทศ: {list(TER)}")

## 1. ตัวแปรภูมิประเทศบนกริด 1 km

- ความลาดชัน: $\theta = \arctan\sqrt{(\partial h/\partial x)^2 + (\partial h/\partial y)^2}$
- TPI (Topographic Position Index) = $h - \overline{h}_{11\times 11\ \text{km}}$ บวก = สันเขา ลบ = หุบ
- **แรงยกจากภูมิประเทศ** ตามการเคลื่อนที่ของพายุ: $w_{\text{oro}} = u\,\dfrac{\partial h}{\partial x} + v\,\dfrac{\partial h}{\partial y}$ (m/s) บวก = เคลื่อนขึ้นลาด

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(7.2, 2.8), layout="constrained")
for ax, (k, cm, lab, vr) in zip(axs, [("elevation", "terrain", "ความสูง (m)", (0, 1200)), ("slope", "magma_r", "ความลาดชัน (°)", (0, 6)),
                                       ("tpi", "RdBu_r", "TPI (m)", (-150, 150))]):
    im = A.show_grid(ax, np.where(meta["coverage"], TER[k], np.nan), meta, cmap=cm, vmin=vr[0], vmax=vr[1])
    M.add_range_rings(ax, s, rings_km=[100, 160, 240], label=False, name=False)
    ax.set_xlim(-245, 245); ax.set_ylim(-245, 245); ax.set_aspect("equal"); ax.tick_params(labelsize=6)
    cb = fig.colorbar(im, ax=ax, orientation="horizontal", shrink=0.85, pad=0.04); cb.set_label(lab, fontsize=7)
fig.suptitle("ภูมิประเทศ (Copernicus GLO-90 เฉลี่ยเป็น 1 km) และวงระยะ 100/160/240 km", fontsize=9)
M.savefig(fig, "nb05_terrain")
plt.show()

## 2. จุดกำเนิดพายุอยู่บนพื้นที่สูงกว่าที่คาดหรือไม่

$H_0$: ความสูงของจุดกำเนิดเหมือนการสุ่มจุดจากพื้นที่ที่เรดาร์มองเห็น ทดสอบด้วยการสุ่มตำแหน่ง 5,000 ครั้ง (`tt.permutation_test`) แต่ **ต้องสุ่มจากวงระยะเดียวกับจุดกำเนิดจริง** ไม่อย่างนั้นผลจะปนกับการที่เรดาร์ตรวจพายุไกลได้น้อยกว่า

In [ ]:
X, Y = np.meshgrid(meta["x"], meta["y"]); RNG = np.hypot(X, Y) / 1e3
start = C.sort_values("time").groupby("uid").first().loc[T["uid"]]
ci = np.clip(np.round((start["cx"] - meta["x"][0]) / 1000).astype(int), 0, X.shape[1] - 1)
ri = np.clip(np.round((meta["y"][0] - start["cy"]) / 1000).astype(int), 0, X.shape[0] - 1)
h0 = TER["elevation"][ri, ci]
r0 = RNG[ri, ci]
pop_all = TER["elevation"][meta["coverage"]]
pop_ring = TER["elevation"][(RNG >= np.percentile(r0, 10)) & (RNG <= np.percentile(r0, 90))]
for name, pop in [("ทั้งรัศมี", pop_all), ("วงระยะเดียวกับจุดกำเนิด (p10–p90)", pop_ring)]:
    obs, exp, p = tt.permutation_test(h0, pop, n=5000)
    print(f"{name:35s}: มัธยฐานจุดกำเนิด {obs:.0f} m vs คาด {exp:.0f} m, p = {p:.4f}")

## 3. ระยะเรดาร์เป็นตัวแปรแทรกซ้อน: partial Spearman

ต้องการรู้ความสัมพันธ์ระหว่างความสูงภูมิประเทศ ($x$) กับความรุนแรง ($y$) โดยตัดผลของระยะ ($z$) ออก: แปลงเป็นอันดับ แล้วหาส่วนเหลือหลังถดถอยเชิงเส้นกับ $z$

$$\rho_{xy\cdot z} = \operatorname{corr}\big(e_x, e_y\big), \qquad e_x = r_x - \hat r_x(r_z),\ e_y = r_y - \hat r_y(r_z)$$

In [ ]:
rows = []
for y in ["Zmax_obs", "hail_col_km2", "KDP95", "EDR95", "lifetime_min"]:
    for x in ["dem_mean", "slope_mean", "upslope_frac"]:
        d = T[[x, y, "range_km"]].dropna()
        if len(d) < 10:
            continue
        r_raw, p_raw = stats.spearmanr(d[x], d[y])
        r_p, p_p, n = tt.partial_spearman(d, x, y, "range_km")
        rows.append({"x": x, "y": y, "n": n, "ρ": r_raw, "p": p_raw, "ρ | ระยะ": r_p, "p | ระยะ": p_p})
PC = pd.DataFrame(rows)
PC.round(3)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.0), layout="constrained")
sc = axs[0].scatter(T["dem_mean"], T["Zmax_obs"], c=T["range_km"], cmap="viridis", s=12, vmin=0, vmax=240)
axs[0].set_xlabel("ความสูงเฉลี่ยใต้เส้นทาง (m)"); axs[0].set_ylabel("Z สูงสุด (dBZ)")
fig.colorbar(sc, ax=axs[0], label="ระยะจากเรดาร์ (km)")
T["range_class"] = pd.cut(T["range_km"], s["geometry"]["range_bins_km"] + [1e4], labels=["0–100", "100–160", "160–240", ">240"])
tt.boxplot_groups(axs[1], T, "Zmax_obs", "range_class", order=["0–100", "100–160", "160–240"], ylabel="Z สูงสุด (dBZ)",
                  colors=["#edf8b1", "#7fcdbb", "#2c7fb8"])
axs[1].set_xlabel("ระยะจากเรดาร์ (km)")
M.savefig(fig, "nb05_terrain_range_confounding")
plt.show()

## 4. จับคู่เส้นทางกับรายงานความเสียหาย

เงื่อนไขการจับคู่ (ทั้งสองข้อ):
1. **พื้นที่:** footprint ของเส้นทางซ้อนทับ polygon อำเภอที่รายงาน
2. **เวลา:** ช่วงชีวิตของเส้นทาง $[t_{\text{start}}, t_{\text{end}}]$ ซ้อนทับหน้าต่างเวลาของรายงาน (UTC) จากบท 01

รายงานหนึ่งอาจจับคู่ได้หลายเส้นทาง และบางรายงานอาจไม่พบเส้นทางเลย (อยู่นอกรัศมี เรดาร์ไม่มีข้อมูล หรือเวลาที่รายงานคลาดเคลื่อน)

In [ ]:
FP = D.table(f"footprints_{TAG}.gpkg")
FP = FP[FP["uid"].isin(T["uid"])]
REP = gpd.read_file(repo_path("data", "reports", "reports_may2024.gpkg"))
for c in ("win_start_utc", "win_end_utc", "date"):
    REP[c] = pd.to_datetime(REP[c])
REP = REP[REP["date"].isin(T["day"].unique())]                  # เฉพาะวันที่มีผลติดตาม
pairs = A.match_tracks_reports(FP, T, REP)
T["matched"] = T["uid"].isin(pairs["uid"])
print(f"รายงาน {len(REP)} รายการในวันที่มีผลติดตาม → จับคู่ได้ {pairs['report_id'].nunique()} | "
      f"เส้นทางที่จับคู่กับรายงาน {T['matched'].sum()} จาก {len(T)}")

In [ ]:
V = ["Zmax_obs", "hail_col_km2", "KDP95", "EDR95", "DIVmax", "lifetime_min"]
T["กลุ่ม"] = pd.Categorical(np.where(T["matched"], "มีรายงาน", "ไม่มีรายงาน"), ["ไม่มีรายงาน", "มีรายงาน"])
A.effect_table(T, V, "กลุ่ม", order=["ไม่มีรายงาน", "มีรายงาน"]).round(3)

In [ ]:
best = pairs.merge(T[["uid"] + V], on="uid").groupby("report_id")[V].max()
rep_t = REP.set_index("report_id")[["tier", "houses", "r_km"]].join(best, how="inner")
rep_t["tier"] = pd.Categorical(rep_t["tier"], ["น้อย", "กลาง", "มาก"], ordered=True)
fig, axs = plt.subplots(1, 3, figsize=(7.2, 2.8), layout="constrained")
for ax, v in zip(axs, ["Zmax_obs", "hail_col_km2", "DIVmax"]):
    tt.boxplot_groups(ax, rep_t, v, "tier", order=["น้อย", "กลาง", "มาก"], colors=list(M.TIER_COLORS.values()), ylabel=v)
fig.suptitle("ค่าสูงสุดของเส้นทางที่จับคู่ได้ แยกตามระดับความเสียหายของรายงาน (บท 01)", fontsize=9)
M.savefig(fig, "nb05_reports_tier_radar")
plt.show()

## 5. สภาพแวดล้อมรายวัน (sounding 00 UTC) กับพายุรายวัน

แต่ละวันมีค่าเดียว จึงเป็นหน่วยอิสระที่เหมาะกับการหาสหสัมพันธ์ ต้องมีอย่างน้อย ~8 วันขึ้นไปจึงเริ่มตีความได้

In [ ]:
env = pd.read_csv(repo_path("data", "sounding", "may2024_environment_daily.csv"), parse_dates=["day"])
dd = T.groupby("day").agg(n_tracks=("uid", "size"), life_med=("lifetime_min", "median"), hail_max=("hail_col_km2", "max"),
                          Zmax=("Zmax_obs", "max")).reset_index().merge(env, on="day", how="left")
rows = []
for e in ["MLCAPE_Jkg", "DCAPE_Jkg", "PW_mm", "shear_0_6km_ms", "lapse_700_500_Ckm", "z0C_m"]:
    for v in ["n_tracks", "life_med", "hail_max"]:
        d = dd[[e, v]].dropna()
        if len(d) >= 5:
            r, p = stats.spearmanr(d[e], d[v]); rows.append({"สภาพแวดล้อม": e, "พายุ": v, "n วัน": len(d), "ρ": r, "p": p})
print("ต้องมีอย่างน้อย 5 วันที่มีทั้ง sounding ครบและพายุ" if not rows else "")
pd.DataFrame(rows).round(3)

## 6. แบบฝึกหัด

**ข้อ 1** ทำ permutation test ในหัวข้อ 2 ซ้ำด้วย **TPI** แทนความสูง พายุก่อตัวบนสันเขา (TPI บวก) มากกว่าที่คาดหรือไม่

In [ ]:
# เขียนโค้ดของตนเองที่นี่

In [ ]:
# @title เฉลยข้อ 1
tpi0 = TER["tpi"][ri, ci]
pop = TER["tpi"][(RNG >= np.percentile(r0, 10)) & (RNG <= np.percentile(r0, 90))]
print(tt.permutation_test(tpi0, pop, n=5000))

**ข้อ 2** ขยายเงื่อนไขการจับคู่ด้วย buffer 10 km รอบอำเภอ (`A.match_tracks_reports(..., buffer_km=10)`) จำนวนรายงานที่จับคู่ได้เพิ่มขึ้นเท่าไร และเส้นทางที่จับคู่เพิ่มมามีลักษณะต่างจากเดิมหรือไม่

In [ ]:
# เขียนโค้ดของตนเองที่นี่

In [ ]:
# @title เฉลยข้อ 2
p10 = A.match_tracks_reports(FP, T, REP, buffer_km=10)
print(f"รายงานที่จับคู่ได้: {pairs['report_id'].nunique()} → {p10['report_id'].nunique()} | เส้นทาง: {pairs['uid'].nunique()} → {p10['uid'].nunique()}")
new = set(p10["uid"]) - set(pairs["uid"])
print(T[T["uid"].isin(new)][["Zmax_obs", "hail_col_km2", "range_km"]].median())

**ข้อ 3** ใช้คอลัมน์ `w_oro_ms` ในตารางเซลล์ เปรียบเทียบ Z สูงสุดของเซลล์ที่กำลังเคลื่อนขึ้นลาด (w > 0.05 m/s) กับลงลาด (w < −0.05 m/s) เฉพาะเซลล์ในวงระยะ 100–200 km (ควบคุมระยะ)

In [ ]:
# เขียนโค้ดของตนเองที่นี่

In [ ]:
# @title เฉลยข้อ 3
cc = C[C["uid"].isin(T["uid"]) & C["observed"] & C["range_km"].between(100, 200)]
up, dn = cc.loc[cc["w_oro_ms"] > 0.05, "Zmax"], cc.loc[cc["w_oro_ms"] < -0.05, "Zmax"]
print(f"ขึ้นลาด n={len(up)} มัธยฐาน {up.median():.1f} | ลงลาด n={len(dn)} มัธยฐาน {dn.median():.1f}")
if min(len(up), len(dn)) > 5:
    print(stats.mannwhitneyu(up, dn), "Cliff δ =", round(tt.cliffs_delta(up, dn), 2))

## 7. คำถามชวนคิด
1. ถ้าพายุก่อตัวบนภูเขามากกว่าที่คาด แต่ภูเขาอยู่ไกลเรดาร์ เรามั่นใจได้แค่ไหนว่าไม่ได้เกิดจากการที่เรดาร์ "เห็น" พายุบนภูเขาได้ดีกว่า (เช่น ลำคลื่นถูกบดบังน้อยกว่า)
2. รายงานที่จับคู่ไม่ได้เลยเกิดจากสาเหตุใดได้บ้าง และจะจำแนกสาเหตุแต่ละแบบด้วยข้อมูลที่มีอย่างไร (`coverage`, `radar_verifiable`, เวลาแบบคำ)
3. sounding 07 น. ที่โคราชเป็นตัวแทนสภาพแวดล้อมของพายุ 17 น. ที่เลยได้ดีแค่ไหน

## 8. แนวทางต่อยอด
- ใช้ ERA5 รายชั่วโมง (CAPE, ลมระดับต่ำ) แทน sounding เพื่อได้สภาพแวดล้อมตามตำแหน่งและเวลาของพายุ
- วิเคราะห์ทิศทางลมระดับต่ำเทียบแนวเขา (upslope flow) เป็นกลไกการก่อตัว
- สร้างแบบจำลองความน่าจะเป็นของความเสียหาย (logistic/GAM) จากตัวแปรเรดาร์ + ความหนาแน่นประชากร

**อ้างอิง:** Houze (2012) *Rev. Geophys.* 50, RG1001; Wilks (2019) *Statistical Methods in the Atmospheric Sciences*; Lakshmanan et al. (2009) *J. Appl. Meteor. Climatol.* 48, 2531–2549